In [ ]:
!pip install osmnx geopandas folium matplotlib shapely
import requests

In [ ]:
import osmnx as ox

buildings = ox.features_from_place(
    "Maxvorstadt, Munich, Germany",
    tags={"building": True}
)

print(type(buildings))
print(len(buildings))

In [ ]:
print(buildings.columns.tolist())
print(buildings.head(3))

In [ ]:
from shapely.geometry import Polygon, MultiPolygon

buildings_poly = buildings[
    buildings.geometry.apply(lambda g: isinstance(g, (Polygon, MultiPolygon)))
].copy()

print(f"قبل التصفية: {len(buildings)}")
print(f"بعد التصفية: {len(buildings_poly)}")

In [ ]:
buildings_proj = buildings_poly.to_crs(epsg=32632)

buildings_proj["area_m2"] = buildings_proj.geometry.area

print(buildings_proj[["area_m2"]].head())
print(f"متوسط المساحة: {buildings_proj['area_m2'].mean():.1f} م²")
print(f"أكبر سطح: {buildings_proj['area_m2'].max():.1f} م²")
print(f"أصغر سطح: {buildings_proj['area_m2'].min():.1f} م²")

In [ ]:
MIN_AREA = 50

good_buildings = buildings_proj[buildings_proj["area_m2"] >= MIN_AREA].copy()

print(f"قبل: {len(buildings_proj)} مبنى")
print(f"بعد: {len(good_buildings)} مبنى")
print(f"اتشال: {len(buildings_proj) - len(good_buildings)} مبنى صغير")

In [ ]:
PANEL_AREA = 2
PANEL_POWER = 400
EFFICIENCY = 0.60

good_buildings["usable_area"] = good_buildings["area_m2"] * EFFICIENCY

good_buildings["num_panels"] = (good_buildings["usable_area"] // PANEL_AREA).astype(int)

good_buildings["power_kw"] = good_buildings["num_panels"] * PANEL_POWER / 1000

print(good_buildings[["area_m2", "usable_area", "num_panels", "power_kw"]].head(10))

In [ ]:

url = "https://re.jrc.ec.europa.eu/api/v5_3/PVcalc"
params = {
    "lat": 48.14,         
    "lon": 11.57,
    "peakpower": 1,    
    "loss": 15,         
    "outputformat": "json"
}

response = requests.get(url, params=params)
data = response.json()
annual = data["outputs"]["totals"]["fixed"]["E_y"]
print(f"الإنتاج السنوي لكل كيلوواط: {annual} كيلوواط ساعة")

In [ ]:
import requests

url = "https://re.jrc.ec.europa.eu/api/v5_3/PVcalc"
params = {
    "lat": 48.14,
    "lon": 11.57,
    "peakpower": 1,
    "loss": 15,
    "outputformat": "json"
}

response = requests.get(url, params=params)
data = response.json()
API_ANNUAL_PER_KW = data["outputs"]["totals"]["fixed"]["E_y"]

print(f"الإنتاج السنوي لكل كيلوواط: {API_ANNUAL_PER_KW} كيلوواط ساعة")

good_buildings["annual_kwh"] = good_buildings["power_kw"] * API_ANNUAL_PER_KW

def classify(row):
    if row["annual_kwh"] >= 10000:
        return "ممتاز 🟢"
    elif row["annual_kwh"] >= 4000:
        return "متوسط 🟡"
    else:
        return "ضعيف 🔴"

good_buildings["category"] = good_buildings.apply(classify, axis=1)

In [ ]:
print("=" * 45)
print("☀️ تقرير الطاقة الشمسية - Maxvorstadt, Munich")
print("=" * 45)

print(f"\n📊 إجمالي المباني الصالحة: {len(good_buildings)}")
print(f"📐 إجمالي المساحة: {good_buildings['area_m2'].sum():,.0f} م²")
print(f"🔲 إجمالي الألواح: {good_buildings['num_panels'].sum():,.0f} لوح")
print(f"⚡ إجمالي القدرة: {good_buildings['power_kw'].sum():,.0f} كيلوواط")
print(f"🔋 إجمالي الإنتاج: {good_buildings['annual_kwh'].sum():,.0f} كيلوواط ساعة/سنة")

print(f"\n🏠 ده بيكفي تقريباً {int(good_buildings['annual_kwh'].sum() / 3500)} بيت في السنة")

print(f"\n📈 التصنيف:")
print(good_buildings["category"].value_counts().to_string())

In [ ]:
import folium

good_buildings_wgs = good_buildings.to_crs(epsg=4326)

bounds = good_buildings_wgs.total_bounds   # [minlon, minlat, maxlon, maxlat]

m = folium.Map(location=[48.14, 11.57], zoom_start=14)

colors = {"ممتاز 🟢": "green", "متوسط 🟡": "orange", "ضعيف 🔴": "red"}

def style_func(feature):
    cat = feature['properties']['category']
    c = colors.get(cat, 'gray')
    return {"fillColor": c, "color": c, "weight": 1, "fillOpacity": 0.6}

folium.GeoJson(
    good_buildings_wgs,
    style_function=style_func,
    tooltip=folium.GeoJsonTooltip(
        fields=['area_m2', 'annual_kwh', 'category'],
        aliases=['مساحة م²', 'إنتاج', 'التصنيف']
    )
).add_to(m)

m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

m

In [ ]:
import json, numpy as np, os
from shapely.geometry import mapping

# ═══════════════════════════════════════════════════════════
# 1) NEUTRAL KEYS + COLORS + DATA PROCESSING
# ═══════════════════════════════════════════════════════════
KEY_OF = {"ممتاز 🟢": "excellent", "متوسط 🟡": "moderate", "ضعيف 🔴": "weak"}
COLORS = {"excellent": "#f5a623", "moderate": "#e8632a", "weak": "#4fb0a8"}
ORDER  = ["excellent", "moderate", "weak"]
EXCELLENT_THR = 10000

# افترض أن good_buildings موجود مسبقاً في الـ Notebook
gb = good_buildings.copy()
gb["cat_key"] = gb["category"].map(KEY_OF)

n_buildings   = int(len(gb))
total_panels  = int(gb["num_panels"].sum())
capacity_mw   = gb["power_kw"].sum() / 1000
annual_gwh    = gb["annual_kwh"].sum() / 1_000_000
homes_powered = int(gb["annual_kwh"].sum() / 3500)

cat_counts = gb["cat_key"].value_counts()
categories = [{"key": k, "count": int(cat_counts.get(k, 0)), "color": COLORS[k]} for k in ORDER]

# ---- (1) LORENZ + GINI ----
_vals = np.sort(gb["annual_kwh"].values)
_cum  = np.cumsum(_vals)
_lo_y = np.concatenate([[0.0], _cum / _cum[-1]])
_lo_x = np.linspace(0, 1, len(_vals) + 1)
_trapz = getattr(np, "trapezoid", None) or np.trapz
_gini  = round(float(1 - 2 * _trapz(_lo_y, _lo_x)), 3)
_step  = max(1, (len(_lo_x)) // 64)
lorenz = {"x": (_lo_x * 100)[::_step].round(1).tolist(), "y": (_lo_y * 100)[::_step].round(1).tolist()}

_sorted = np.sort(gb["annual_kwh"].values)[::-1]
_ccum   = np.cumsum(_sorted) / _sorted.sum() * 100
top20_share = round(float(_ccum[min(int(len(_ccum) * 0.2), len(_ccum) - 1)]), 1)

# ---- (2) YIELD BY BUILDING TYPE ----
_CLEAN = {"yes": "unspecified", "house": "house", "residential": "residential",
          "apartments": "apartments", "commercial": "commercial", "retail": "retail",
          "industrial": "industrial", "office": "office", "garage": "garage",
          "roof": "roof", "terrace": "terrace", "school": "school", "hospital": "hospital"}
by_type = []
if "building" in gb.columns:
    _bt = gb.copy()
    _bt["btype"] = _bt["building"].fillna("unspecified").astype(str).map(
        lambda v: _CLEAN.get(v, v if v != "yes" else "unspecified"))
    _bt = _bt[~_bt["btype"].isin(["unspecified", "unknown", "None", ""])]
    if len(_bt):
        _g = _bt.groupby("btype").agg(count=("annual_kwh", "size"),
                                       output=("annual_kwh", "sum")).reset_index()
        _g = _g.sort_values("output", ascending=False).head(8)
        by_type = [{"type": r["btype"], "count": int(r["count"]),
                    "output": round(r["output"])} for _, r in _g.iterrows()]

# ---- (3) SPATIAL DENSITY (✅ تم حل مشكلة الـ CRS Warning هنا) ----
_wgs = gb.to_crs(epsg=4326)
_sp  = gb.sample(min(1500, len(gb)), random_state=11)

# الحل الجذري: تحويل لـ Projected CRS (UTM Zone 32N لميونخ) -> حساب الـ Centroid -> العودة لـ WGS84
_geom_wgs = _wgs.loc[_sp.index].geometry
_geom_proj = _geom_wgs.to_crs(epsg=32632)
_sc_proj = _geom_proj.centroid
_sc = _sc_proj.to_crs(epsg=4326)

spatial = {"lon": _sc.x.round(4).tolist(), "lat": _sc.y.round(4).tolist(),
           "out": _sp["annual_kwh"].round().tolist()}

# ---- (4) OUTPUT DISTRIBUTION ----
_out = gb["annual_kwh"].values
_clip = np.percentile(_out, 99)
_hc, _he = np.histogram(np.clip(_out, 0, _clip), bins=32)
hist = {"bins": [round((_he[i] + _he[i+1]) / 2) for i in range(len(_hc))],
        "counts": _hc.tolist()}
median_out = round(float(np.median(_out)))

# ---- (5) MARGINAL BUILD-OUT ----
_desc = gb.sort_values("annual_kwh", ascending=False)
_cmw  = (_desc["power_kw"].cumsum() / 1000).values
_mstep = max(1, len(_cmw) // 64)
marginal = {"x": np.arange(1, len(_cmw) + 1)[::_mstep].tolist(),
            "y": _cmw[::_mstep].round(1).tolist()}
_n10 = max(1, int(len(_cmw) * 0.1))
marginal_note = {"n": _n10, "mw": round(float(_cmw[_n10 - 1]), 1)}

# ---- (6) TOP PRODUCERS ----
_top = gb.nlargest(12, "annual_kwh")
top = [{"rank": i+1, "area": round(r["area_m2"]), "panels": int(r["num_panels"]),
        "power_kw": round(r["power_kw"], 1), "output_kwh": round(r["annual_kwh"]),
        "cat": r["cat_key"]} for i, (_, r) in enumerate(_top.iterrows())]

avg_output = gb["annual_kwh"].mean()
pct_good   = gb["cat_key"].eq("excellent").mean() * 100
med_eff    = (gb["annual_kwh"] / gb["area_m2"].clip(lower=1)).median()

DATA = {
    "meta": {"place": "Maxvorstadt, Munich", "source": "OpenStreetMap + EU PVGIS"},
    "kpis": [
        {"ico": "🏢", "value": n_buildings, "suffix": "", "label": "buildings", "accent": "sun"},
        {"ico": "🔲", "value": total_panels, "suffix": "", "label": "panels", "accent": "heat"},
        {"ico": "⚡", "value": round(capacity_mw), "suffix": " MW", "label": "capacity", "accent": "teal"},
        {"ico": "🔋", "value": round(annual_gwh), "suffix": " GWh", "label": "per year", "accent": "sun"},
        {"ico": "🏠", "value": homes_powered, "suffix": "", "label": "homes", "accent": "heat"},
    ],
    "glance": {"eff": f"{med_eff:,.1f}", "top20": top20_share, "avg": f"{avg_output:,.0f}", "exc": round(pct_good, 1)},
    "categories": categories, "cat_colors": COLORS,
    "gini": _gini, "lorenz": lorenz, "top20_share": top20_share,
    "by_type": by_type, "spatial": spatial,
    "hist": hist, "median_out": median_out, "excellent_thr": EXCELLENT_THR,
    "marginal": marginal, "marginal_note": marginal_note,
    "top": top,
    "guide": [
        {"icon": "📉", "en": {"t": "Inequality (Lorenz)", "x": "Gini = {v}. The curve bows away from the equality line."}, "de": {"t": "Ungleichheit", "x": "Gini = {v}. Die Kurve weicht von der Gleichheit ab."}, "v": _gini},
        {"icon": "🏗️", "en": {"t": "By building type", "x": "Real OSM tags, not area. See which use-class carries the yield."}, "de": {"t": "Nach Gebäudetyp", "x": "Echte OSM-Tags, nicht Fläche."}, "v": ""},
        {"icon": "📍", "en": {"t": "Spatial concentration", "x": "Each dot is a roof; brighter = higher yield."}, "de": {"t": "Räumliche Konzentration", "x": "Jeder Punkt ein Dach; heller = mehr Ertrag."}, "v": ""},
        {"icon": "📊", "en": {"t": "Output distribution", "x": "Median {v} kWh. Most roofs are modest, long tail of giants."}, "de": {"t": "Ertragsverteilung", "x": "Median {v} kWh. Die meisten Dächer sind moderat."}, "v": f"{median_out:,}"},
        {"icon": "🚀", "en": {"t": "Marginal build-out", "x": "Install best {n} roofs first → {mw} MW. Diminishing returns."}, "de": {"t": "Marginaler Ausbau", "x": "Zuerst die besten {n} Dächer → {mw} MW."}, "v": ""},
        {"icon": "🏆", "en": {"t": "Top producers", "x": "The rooftops worth a site visit tomorrow."}, "de": {"t": "Top-Erzeuger", "x": "Die Dächer, die morgen einen Vor-Ort-Termin wert sind."}, "v": ""},
    ],
}

with open("data.js", "w", encoding="utf-8") as f:
    f.write("window.SOLAR_DATA = " + json.dumps(DATA, ensure_ascii=False) + ";\n")

# 1b) BUILDINGS
feats = []
for (idx, row), (widx, wrow) in zip(gb.iterrows(), _wgs.iterrows()):
    feats.append({"type": "Feature",
                  "properties": {"o": round(row["annual_kwh"]), "c": row["cat_key"]},
                  "geometry": mapping(wrow.geometry.simplify(0.00004))})
with open("buildings.js", "w", encoding="utf-8") as f:
    f.write("window.BUILDINGS = " + json.dumps({"type": "FeatureCollection", "features": feats}, ensure_ascii=False) + ";\n")

# ═══════════════════════════════════════════════════════════
# 2) STYLES (✅ تمت إضافة أيقونات Sidebar + زر Reset)
# ═══════════════════════════════════════════════════════════
CSS = r"""
:root{
  --bg:#14110d; --surface:#1c1813; --surface-2:#241f18;
  --line:rgba(242,237,228,0.08); --line-2:rgba(242,237,228,0.16);
  --text:#f2ede4; --muted:#9b9384; --faint:#6b6457;
  --sun:#f5a623; --heat:#e8632a; --teal:#4fb0a8;
  --glow:0 14px 44px -14px rgba(245,166,35,0.45); --r:16px;
}
[data-theme="light"]{
  --bg:#f3efe7; --surface:#fbf9f4; --surface-2:#efe9dd;
  --line:rgba(26,23,20,0.09); --line-2:rgba(26,23,20,0.16);
  --text:#1a1714; --muted:#6b6457; --faint:#9b9384;
  --glow:0 14px 44px -18px rgba(232,99,42,0.32);
}
*{margin:0;padding:0;box-sizing:border-box;}
body{font-family:'DM Sans',system-ui,sans-serif;background:var(--bg);color:var(--text);
  height:100dvh;overflow:hidden;line-height:1.5;
  background-image:radial-gradient(circle at 90% -8%, rgba(245,166,35,0.16), transparent 40%),
  repeating-radial-gradient(circle at 90% -8%, transparent 0 40px, rgba(245,166,35,0.045) 40px 41px);}
h1,h2,h3,.display{font-family:'Space Grotesk',sans-serif;letter-spacing:-0.02em;}
.serif{font-family:'Instrument Serif',serif;font-style:italic;font-weight:400;}
.mono{font-variant-numeric:tabular-nums;}
.app{display:grid;grid-template-columns:236px 1fr;height:100dvh;}
.side{background:var(--surface);border-right:1px solid var(--line);padding:24px 16px;display:flex;flex-direction:column;min-height:0;}
.brand{display:flex;align-items:baseline;gap:7px;}
.brand .b1{font-family:'Space Grotesk';font-weight:700;font-size:20px;}
.brand .b2{font-family:'Instrument Serif';font-style:italic;color:var(--sun);font-size:23px;}
.brand-sub{color:var(--faint);font-size:10px;letter-spacing:2px;text-transform:uppercase;margin:4px 0 30px;}
.nav{display:flex;flex-direction:column;gap:2px;}

/* ✅ أيقونات Sidebar الجديدة */
.nav button{all:unset;cursor:pointer;display:flex;align-items:center;gap:12px;padding:11px 14px;border-radius:11px;color:var(--muted);font-weight:500;font-size:14px;transition:.22s;}
.nav-icon{width:20px;height:20px;flex-shrink:0;color:var(--faint);transition:.22s;}
.nav button:hover{color:var(--text);background:var(--surface-2);}
.nav button:hover .nav-icon{color:var(--sun);transform:scale(1.15) translateX(2px);}
.nav button.active{color:var(--text);background:var(--surface-2);border-left:3px solid var(--sun);}
.nav button.active .nav-icon{color:var(--sun);}

.side-foot{margin-top:auto;color:var(--faint);font-size:10.5px;line-height:1.7;}
.main{display:flex;flex-direction:column;min-height:0;padding:18px 30px 22px;}
.topbar{flex:0 0 auto;display:flex;justify-content:space-between;align-items:center;background:rgba(28,24,19,0.55);backdrop-filter:blur(16px);border:1px solid var(--line);border-radius:var(--r);padding:10px 18px;margin-bottom:16px;}
[data-theme="light"] .topbar{background:rgba(251,249,244,0.72);}
.crumb{color:var(--muted);font-size:13px;} .crumb b{color:var(--text);font-weight:600;}
.actions{display:flex;gap:8px;align-items:center;}
.lang{display:flex;border:1px solid var(--line);border-radius:9px;overflow:hidden;}
.lang button{all:unset;cursor:pointer;padding:7px 11px;font-size:12px;font-weight:700;color:var(--muted);transition:.2s;}
.lang button.on{background:var(--sun);color:#1a1206;}
.ibtn{width:36px;height:36px;border-radius:9px;border:1px solid var(--line);background:var(--surface-2);color:var(--text);cursor:pointer;font-size:15px;transition:.22s;}
.ibtn:hover{border-color:var(--sun);color:var(--sun);transform:translateY(-2px);}
.view{flex:1 1 auto;min-height:0;display:none;flex-direction:column;}
.view.active{display:flex;animation:fade .4s ease;}
@keyframes fade{from{opacity:0;transform:translateY(8px);}to{opacity:1;transform:none;}}
.kicker{color:var(--sun);font-size:11px;letter-spacing:2.5px;text-transform:uppercase;font-weight:600;}
.hero{flex:0 0 auto;display:grid;grid-template-columns:1.4fr 1fr;gap:26px;align-items:center;border:1px solid var(--line);border-radius:20px;padding:30px 36px;margin-bottom:16px;background:linear-gradient(115deg,var(--surface),transparent);position:relative;overflow:hidden;}
.hero::after{content:'';position:absolute;right:-40px;top:-40px;width:240px;height:240px;border-radius:50%;background:radial-gradient(circle,rgba(245,166,35,0.22),transparent 70%);}
.hero h1{font-size:38px;line-height:1.04;margin:10px 0 12px;position:relative;}
.hero p{color:var(--muted);max-width:48ch;position:relative;}
.hero .bignum{text-align:right;position:relative;}
.hero .bignum .n{font-family:'Instrument Serif';font-style:italic;font-size:96px;line-height:.82;color:var(--sun);text-shadow:0 0 60px rgba(245,166,35,0.35);}
.hero .bignum .u{color:var(--muted);font-size:12px;letter-spacing:2px;text-transform:uppercase;}
.kpis{flex:0 0 auto;display:grid;grid-template-columns:repeat(5,1fr);gap:13px;margin-bottom:16px;}
.kpi{background:var(--surface);border:1px solid var(--line);border-radius:var(--r);padding:18px;position:relative;overflow:hidden;transition:.28s;}
.kpi:hover{transform:translateY(-5px);box-shadow:var(--glow);border-color:var(--line-2);}
.kpi .stripe{position:absolute;left:0;top:0;bottom:0;width:3px;}
.kpi[data-a="sun"] .stripe{background:var(--sun);} .kpi[data-a="heat"] .stripe{background:var(--heat);} .kpi[data-a="teal"] .stripe{background:var(--teal);}
.kpi .ico{font-size:18px;opacity:.85;} .kpi .num{font-family:'Space Grotesk';font-size:28px;font-weight:700;margin:8px 0 2px;}
.kpi .lbl{color:var(--muted);font-size:11.5px;text-transform:uppercase;letter-spacing:.5px;}
.glance{flex:1 1 auto;min-height:0;display:grid;grid-template-columns:1.1fr 1fr;gap:16px;}
.glance .card{background:var(--surface);border:1px solid var(--line);border-radius:var(--r);padding:24px 26px;display:flex;flex-direction:column;justify-content:center;min-height:0;}
.glance .lead{font-family:'Instrument Serif';font-style:italic;font-size:30px;line-height:1.18;color:var(--text);}
.glance .lead em{color:var(--sun);font-style:italic;}
.minis{display:grid;grid-template-columns:1fr 1fr;gap:12px;}
.mini{background:var(--surface-2);border:1px solid var(--line);border-radius:12px;padding:16px;}
.mini .v{font-family:'Space Grotesk';font-size:24px;font-weight:700;color:var(--sun);}
.mini .k{color:var(--muted);font-size:12px;margin-top:3px;}
.map-head{flex:0 0 auto;display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:12px;gap:12px;}
.map-head h2{font-size:24px;}
.map-stage{flex:1 1 auto;min-height:0;position:relative;border:1px solid var(--line);border-radius:var(--r);overflow:hidden;}
#map{position:absolute;inset:0;}
.float{position:absolute;z-index:500;background:rgba(20,17,13,0.82);backdrop-filter:blur(12px);border:1px solid var(--line-2);border-radius:14px;padding:14px 16px;box-shadow:var(--glow);}
[data-theme="light"] .float{background:rgba(251,249,244,0.86);}
.float.stats{top:14px;left:14px;min-width:210px;}
.float.stats .row{display:flex;justify-content:space-between;gap:18px;font-size:13px;padding:3px 0;}
.float.stats .row b{font-family:'Space Grotesk';color:var(--sun);}
.float.slider{bottom:16px;left:50%;transform:translateX(-50%);width:min(440px,80%);}
.float.slider label{font-size:11px;letter-spacing:1px;text-transform:uppercase;color:var(--muted);display:flex;justify-content:space-between;}
.float.slider label b{color:var(--sun);font-family:'Space Grotesk';}
.float.slider input[type=range]{width:100%;margin-top:8px;accent-color:var(--sun);height:4px;}
.float.legend{top:14px;right:14px;font-size:12px;}
.float.legend .li{display:flex;align-items:center;gap:9px;padding:4px 0;}
.float.legend .sw{width:12px;height:12px;border-radius:4px;}
.float.legend .li b{margin-left:auto;font-family:'Space Grotesk';}
.map-tools{position:absolute;top:14px;right:14px;z-index:600;display:flex;flex-direction:column;gap:8px;}
.map-tools button{width:38px;height:38px;border-radius:10px;border:1px solid var(--line-2);background:rgba(20,17,13,0.82);backdrop-filter:blur(10px);color:var(--text);cursor:pointer;font-size:16px;transition:.2s;}
.map-tools button:hover{border-color:var(--sun);color:var(--sun);}

/* ✅ تنسيق زر Reset Charts */
.a-head{flex:0 0 auto;display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:12px;}
.a-head h2{font-size:24px;}
.reset-btn{all:unset;cursor:pointer;display:flex;align-items:center;gap:8px;padding:9px 16px;background:rgba(245,166,35,0.1);border:1px solid rgba(245,166,35,0.4);border-radius:9px;color:var(--sun);font-size:13px;font-weight:600;transition:.25s;}
.reset-btn:hover{background:var(--sun);color:#1a1206;transform:translateY(-2px);box-shadow:0 4px 12px rgba(245,166,35,0.25);}
.reset-btn svg{width:16px;height:16px;transition:transform .5s ease;}
.reset-btn:hover svg{transform:rotate(-180deg);}
.reset-btn.resetting svg{animation:spin .6s ease;}
@keyframes spin{from{transform:rotate(0deg);}to{transform:rotate(-360deg);}}

.a-grid{flex:1 1 auto;min-height:0;display:grid;grid-template-columns:repeat(3,1fr);grid-template-rows:minmax(0,1fr) minmax(0,1fr);gap:13px;}
.cell{background:var(--surface);border:1px solid var(--line);border-radius:var(--r);padding:15px 18px;display:flex;flex-direction:column;min-height:0;transition:.25s;}
.cell:hover{border-color:var(--line-2);}
.cell .ct{font-size:16px;font-weight:600;letter-spacing:-0.01em;}
.cell .cw{color:var(--muted);font-size:12.5px;margin:3px 0 7px;line-height:1.4;}
.cell .plot{flex:1 1 auto;min-height:0;width:100%;}
.read-strip{flex:0 0 auto;display:flex;gap:11px;overflow-x:auto;margin-top:13px;padding-bottom:4px;}
.read-strip::-webkit-scrollbar{height:6px;} .read-strip::-webkit-scrollbar-thumb{background:var(--line-2);border-radius:6px;}
.read-strip .it{flex:1 1 0;min-width:182px;background:var(--surface);border:1px solid var(--line);border-radius:13px;padding:12px 14px;display:flex;gap:11px;align-items:flex-start;transition:.22s;}
.read-strip .it:hover{border-color:var(--sun);transform:translateY(-3px);}
.read-strip .ic{font-size:18px;line-height:1.2;}
.read-strip .tt{font-weight:600;font-size:12.5px;} .read-strip .xx{color:var(--muted);font-size:11.5px;line-height:1.4;margin-top:2px;}
@media(max-width:980px){
  body{height:auto;overflow:auto;} .app{grid-template-columns:1fr;height:auto;}
  .side{position:fixed;left:0;right:0;bottom:0;top:auto;height:auto;flex-direction:row;border-right:0;border-top:1px solid var(--line);padding:8px;z-index:800;justify-content:space-around;}
  .brand,.brand-sub,.side-foot{display:none;} .nav{flex-direction:row;gap:4px;width:100%;}
  .nav button{flex:1;justify-content:center;font-size:12px;padding:9px 4px;}
  .main{height:auto;overflow:visible;padding:14px 14px 90px;}
  .view{height:auto;overflow:visible;} .view.active{display:flex;}
  .kpis{grid-template-columns:repeat(2,1fr);} .hero,.glance,.a-grid{grid-template-columns:1fr;}
  .a-grid{grid-template-rows:none;} .cell .plot{height:260px;} .hero .bignum{text-align:left;}
  .read-strip{flex-wrap:wrap;overflow:visible;} .read-strip .it{min-width:46%;}
}
"""
with open("styles.css", "w", encoding="utf-8") as f:
    f.write(CSS)

# ═══════════════════════════════════════════════════════════
# 3) APP.JS (✅ تمت إضافة دالة resetCharts)
# ═══════════════════════════════════════════════════════════
JS = r"""
const D = window.SOLAR_DATA;
const $ = s => document.querySelector(s);
const $$ = s => document.querySelectorAll(s);
let LANG = localStorage.getItem('solar-lang') || 'en';
const T = k => (I18N[LANG] && I18N[LANG][k]) || I18N.en[k] || k;
const fmt = n => Number(n).toLocaleString(LANG==='de'?'de-DE':'en-US');

const I18N = {
 en:{ overview:'Overview', map:'Spatial Map', analytics:'Analytics',
   crumb_pre:'Solar·München / ', brand_sub:'PV Potential Observatory',
   hero_tag:'ROOF-BY-ROOF ASSESSMENT', hero_h:'How much sun is<br>sleeping on these roofs?',
   hero_p:'A spatial photovoltaic audit of every qualified building in Maxvorstadt.',
   hero_u:'GWh / year unlockable',
   k_buildings:'buildings', k_panels:'panels', k_capacity:'capacity', k_per_year:'per year', k_homes:'homes',
   glance_k:'AT A GLANCE',
   lead:'Rooftops here hide <em>{gwh} GWh</em> a year — clean power for roughly <em>{homes} homes</em>.',
   m_eff:'median output density', m_eff_u:'kWh per m² of roof', m_top:'top 20% of roofs', m_top_u:'of all energy',
   m_avg:'average per roof', m_avg_u:'kWh / year', m_exc:'excellent roofs', m_exc_u:'> 10 MWh / year',
   map_k:'GEOSPATIAL', map_h:'Interactive Building Map',
   s_showing:'Showing', s_buildings:'buildings', s_output:'their output', s_of:'of', s_sample:'sampled roofs',
   sl_label:'Minimum annual output', lg_title:'Solar class',
   a_k:'DISTRIBUTION', a_h:'Six readings that actually mean something', read_t:'READING GUIDE',
   c_lorenz:'Inequality — Lorenz & Gini', w_lorenz:'How far the yield curve bows from perfect equality.',
   c_type:'Yield by building type', w_type:'Real OSM use-class, not roof area.',
   c_spatial:'Where the power lives', w_spatial:'Each dot a roof; brighter dots = higher annual yield.',
   c_dist:'Output distribution', w_dist:'The shape of the harvest — most roofs modest, a long tail of giants.',
   c_marg:'Marginal build-out', w_marg:'Cumulative capacity as you install the best roofs first.',
   c_top:'Top producers', w_top:'The rooftops worth a site visit tomorrow.',
   ax_roofs:'% of roofs', ax_output_pct:'% of total output', ax_type_x:'annual output (kWh)',
   ax_lon:'longitude', ax_lat:'latitude', ax_dist_x:'annual output (kWh)', ax_count:'buildings',
   ax_marg_x:'roofs installed (best first)', ax_marg_y:'cumulative capacity (MW)', ax_kwh:'kWh / year',
   cat_excellent:'Excellent', cat_moderate:'Moderate', cat_weak:'Weak',
   foot:'Data: OpenStreetMap · Radiation: EU PVGIS · Engine: Shapely + GeoPandas' },
 de:{ overview:'Übersicht', map:'Karte', analytics:'Analytik',
   crumb_pre:'Solar·München / ', brand_sub:'PV-Potenzial-Observatorium',
   hero_tag:'DACH-FÜR-DACH-ANALYSE', hero_h:'Wie viel Sonne schläft<br>auf diesen Dächern?',
   hero_p:'Ein räumliches Photovoltaik-Audit jedes qualifizierten Gebäudes in der Maxvorstadt.',
   hero_u:'GWh / Jahr erschließbar',
   k_buildings:'Gebäude', k_panels:'Module', k_capacity:'Leistung', k_per_year:'pro Jahr', k_homes:'Haushalte',
   glance_k:'AUF EINEN BLICK',
   lead:'Die Dächer hier verbergen <em>{gwh} GWh</em> pro Jahr — saubere Energie für rund <em>{homes} Haushalte</em>.',
   m_eff:'mittlere Ertragsdichte', m_eff_u:'kWh pro m² Dach', m_top:'oberste 20% der Dächer', m_top_u:'der gesamten Energie',
   m_avg:'Durchschnitt pro Dach', m_avg_u:'kWh / Jahr', m_exc:'exzellente Dächer', m_exc_u:'> 10 MWh / Jahr',
   map_k:'GEORÄUMLICH', map_h:'Interaktive Gebäudekarte',
   s_showing:'Anzeige', s_buildings:'Gebäude', s_output:'ihr Ertrag', s_of:'von', s_sample:'erfassten Dächern',
   sl_label:'Minimaler Jahresertrag', lg_title:'Solarklasse',
   a_k:'VERTEILUNG', a_h:'Sechs Lesarten, die wirklich etwas bedeuten', read_t:'LESEHILFE',
   c_lorenz:'Ungleichheit — Lorenz & Gini', w_lorenz:'Wie stark die Ertragskurve von der Gleichheit abweicht.',
   c_type:'Ertrag nach Gebäudetyp', w_type:'Echte OSM-Nutzungsklasse, nicht Dachfläche.',
   c_spatial:'Wo die Energie lebt', w_spatial:'Jeder Punkt ein Dach; hellere Punkte = höherer Jahresertrag.',
   c_dist:'Ertragsverteilung', w_dist:'Die Form der Ernte — die meisten Dächer moderat, ein langer Schwanz von Riesen.',
   c_marg:'Marginaler Ausbau', w_marg:'Kumulierte Leistung, wenn du zuerst die besten Dächer baust.',
   c_top:'Top-Erzeuger', w_top:'Die Dächer, die morgen einen Vor-Ort-Termin wert sind.',
   ax_roofs:'% der Dächer', ax_output_pct:'% des Gesamtertrags', ax_type_x:'Jahresertrag (kWh)',
   ax_lon:'Längengrad', ax_lat:'Breitengrad', ax_dist_x:'Jahresertrag (kWh)', ax_count:'Gebäude',
   ax_marg_x:'gebaute Dächer (beste zuerst)', ax_marg_y:'kumulierte Leistung (MW)', ax_kwh:'kWh / Jahr',
   cat_excellent:'Exzellent', cat_moderate:'Mittel', cat_weak:'Schwach',
   foot:'Daten: OpenStreetMap · Strahlung: EU PVGIS · Engine: Shapely + GeoPandas' }
};
const catName = k => T('cat_' + k);

function showView(name){
  $$('.view').forEach(v => v.classList.toggle('active', v.dataset.view === name));
  $$('.nav button').forEach(b => b.classList.toggle('active', b.dataset.go === name));
  $('.crumb').innerHTML = T('crumb_pre') + '<b>' + T(name) + '</b>';
  if(name === 'map') setTimeout(initMap, 60);
  if(name === 'analytics'){ if(!window._charts) buildCharts(); else resizeAll(); }
}
$$('.nav button').forEach(b => b.onclick = () => showView(b.dataset.go));

const st = localStorage.getItem('solar-theme'); if(st) document.documentElement.dataset.theme = st;
$('#themeBtn').onclick = () => {
  const t = document.documentElement.dataset.theme === 'light' ? 'dark' : 'light';
  document.documentElement.dataset.theme = t; localStorage.setItem('solar-theme', t);
  if(window._map) setBasemap(); if(window._charts) restyleCharts();
};

function setLang(l){ LANG = l; localStorage.setItem('solar-lang', l);
  $$('.lang button').forEach(b => b.classList.toggle('on', b.dataset.l === l)); applyLang(); }
function applyLang(){
  document.documentElement.lang = LANG;
  $('.brand-sub').textContent = T('brand_sub');
  $$('.nav button').forEach(b => {
      // الحفاظ على الأيقونة وتحديث النص فقط
      const icon = b.querySelector('.nav-icon');
      b.innerHTML = '';
      b.appendChild(icon);
      b.appendChild(document.createTextNode(' ' + T(b.dataset.go)));
  });
  $('#heroTag').textContent = T('hero_tag'); $('#heroH').innerHTML = T('hero_h');
  $('#heroP').textContent = T('hero_p'); $('#heroU').textContent = T('hero_u');
  $('#glanceK').textContent = T('glance_k');
  $('#leadTxt').innerHTML = T('lead').replace('{gwh}', fmt(D.kpis[3].value)).replace('{homes}', fmt(D.kpis[4].value));
  $('#mEffK').textContent = T('m_eff'); $('#mEffU').textContent = T('m_eff_u');
  $('#mTopK').textContent = T('m_top'); $('#mTopU').textContent = T('m_top_u');
  $('#mAvgK').textContent = T('m_avg'); $('#mAvgU').textContent = T('m_avg_u');
  $('#mExcK').textContent = T('m_exc'); $('#mExcU').textContent = T('m_exc_u');
  $('#mapK').textContent = T('map_k'); $('#mapH').textContent = T('map_h');
  $('#slLabel').childNodes[0].nodeValue = T('sl_label') + ' '; $('#lgTitle').textContent = T('lg_title');
  $('#aK').textContent = T('a_k'); $('#aH').textContent = T('a_h'); $('#readT').textContent = T('read_t');
  $$('.ct[data-t]').forEach(e => e.textContent = T(e.dataset.t));
  $$('.cw[data-w]').forEach(e => e.textContent = T(e.dataset.w));
  $('#sShowing').textContent = T('s_showing'); $('#sBld').textContent = T('s_buildings');
  $('#sOutLbl').textContent = T('s_output'); $('#sOf').textContent = T('s_of'); $('#sSample').textContent = T('s_sample');
  renderKpis(); renderLegend(); renderGuide();
  if(window._charts){ window._charts = null; buildCharts(); }
  if(window._map) refreshMapStats();
}

function countUp(el){
  const target = +el.dataset.target, suf = el.dataset.suffix || '', dur = 1400, t0 = performance.now();
  (function tick(now){ const p = Math.min((now-t0)/dur,1), e = 1-Math.pow(1-p,3);
    el.textContent = fmt(Math.floor(target*e)) + suf;
    if(p<1) requestAnimationFrame(tick); else el.textContent = fmt(target) + suf; })(t0);
}
function renderKpis(){
  $('#kpiRow').innerHTML = D.kpis.map(k =>
    `<div class="kpi" data-a="${k.accent}"><div class="stripe"></div><div class="ico">${k.ico}</div>
     <div class="num mono" data-target="${k.value}" data-suffix="${k.suffix}">0</div>
     <div class="lbl">${T('k_'+k.label)}</div></div>`).join('');
  $$('#kpiRow .num').forEach(countUp);
}

function cols(){ const light = document.documentElement.dataset.theme === 'light';
  return {text: light?'#1a1714':'#f2ede4', grid: light?'rgba(0,0,0,0.07)':'rgba(255,255,255,0.07)'}; }
const base = () => { const c = cols();
  return {paper_bgcolor:'rgba(0,0,0,0)', plot_bgcolor:'rgba(0,0,0,0)',
    font:{color:c.text, family:'DM Sans', size:13}, margin:{l:50,r:16,t:12,b:44},
    xaxis:{gridcolor:c.grid, zeroline:false, tickfont:{size:12.5}, titlefont:{size:13}},
    yaxis:{gridcolor:c.grid, zeroline:false, tickfont:{size:12.5}, titlefont:{size:13}}}; };
const CFG = {responsive:true, displayModeBar:false};
const SCALE = [[0,'#4fb0a8'],[0.5,'#f5a623'],[1,'#e8632a']];

function buildCharts(){
  const L = base();
  // (1) LORENZ + GINI
  Plotly.newPlot('pLor',[{x:D.lorenz.x, y:D.lorenz.y, type:'scatter', mode:'lines', fill:'tozeroy', showlegend:false,
     line:{color:'#f5a623', width:2.6}, fillcolor:'rgba(245,166,35,0.13)',
     hovertemplate:'%{x:.0f}% '+T('ax_roofs')+' → %{y:.0f}% '+T('ax_output_pct')+'<extra></extra>'}],
    {...L, xaxis:{...L.xaxis, title:T('ax_roofs'), range:[0,100]}, yaxis:{...L.yaxis, title:T('ax_output_pct'), range:[0,100]},
     shapes:[{type:'line', x0:0, y0:0, x1:100, y1:100, line:{color:'rgba(155,147,132,0.5)', width:1.4, dash:'dot'}}],
     annotations:[{x:62, y:24, text:'Gini = '+D.gini, showarrow:false, font:{size:17, color:'#e8632a', family:'Space Grotesk'}}]}, CFG);
  // (2) BY BUILDING TYPE
  if(D.by_type.length){
    const bt = D.by_type.slice().reverse();
    Plotly.newPlot('pType',[{x:bt.map(b=>b.output), y:bt.map(b=>b.type), type:'bar', orientation:'h', showlegend:false,
      marker:{color:bt.map((_,i)=>i), colorscale:SCALE, showscale:false},
      hovertemplate:'%{y}<br>%{x:,} '+T('ax_kwh')+'<extra></extra>'}],
      {...L, xaxis:{...L.xaxis, title:T('ax_type_x')}, yaxis:{...L.yaxis, automargin:true, tickfont:{size:12.5}}, margin:{...L.margin, l:96}}, CFG);
  } else { $('#pType').innerHTML = '<div style="display:flex;align-items:center;justify-content:center;height:100%;color:var(--faint);font-size:13px">building-type tags unavailable</div>'; }
  // (3) SPATIAL DENSITY
  Plotly.newPlot('pSpa',[{x:D.spatial.lon, y:D.spatial.lat, mode:'markers', type:'scatter', showlegend:false,
    marker:{color:D.spatial.out, colorscale:SCALE, size:5, opacity:0.72, colorbar:{thickness:8, len:0.6, tickfont:{size:10}, outlinewidth:0}},
    hovertemplate:'%{y:.3f}, %{x:.3f}<br>%{marker.color:,} '+T('ax_kwh')+'<extra></extra>'}],
    {...L, xaxis:{...L.xaxis, title:T('ax_lon'), tickformat:'.2f'}, yaxis:{...L.yaxis, title:T('ax_lat'), tickformat:'.2f', scaleanchor:'x'}}, CFG);
  // (4) DISTRIBUTION + median + threshold
  Plotly.newPlot('pDist',[{x:D.hist.bins, y:D.hist.counts, type:'bar', showlegend:false, marker:{color:'#4fb0a8'},
    hovertemplate:'~%{x:,} '+T('ax_kwh')+'<br>%{y} '+T('ax_count')+'<extra></extra>'}],
    {...L, xaxis:{...L.xaxis, title:T('ax_dist_x')}, yaxis:{...L.yaxis, title:T('ax_count')},
     shapes:[{type:'line', x0:D.median_out, x1:D.median_out, y0:0, y1:1, yref:'paper', line:{color:'#f5a623', width:1.8, dash:'dash'}},
             {type:'line', x0:D.excellent_thr, x1:D.excellent_thr, y0:0, y1:1, yref:'paper', line:{color:'#e8632a', width:1.8, dash:'dash'}}],
     annotations:[{x:D.median_out, y:1, yref:'paper', text:'median', showarrow:false, yanchor:'bottom', font:{size:11, color:'#f5a623'}},
                  {x:D.excellent_thr, y:1, yref:'paper', text:'excellent', showarrow:false, yanchor:'bottom', font:{size:11, color:'#e8632a'}}]}, CFG);
  // (5) MARGINAL BUILD-OUT
  Plotly.newPlot('pMarg',[{x:D.marginal.x, y:D.marginal.y, type:'scatter', mode:'lines', fill:'tozeroy', showlegend:false,
    line:{color:'#4fb0a8', width:2.4}, fillcolor:'rgba(79,176,168,0.12)',
    hovertemplate:'%{x:,} roofs → %{y} MW<extra></extra>'}],
    {...L, xaxis:{...L.xaxis, title:T('ax_marg_x')}, yaxis:{...L.yaxis, title:T('ax_marg_y')},
     annotations:[{x:D.marginal_note.n, y:D.marginal_note.mw, text:'top 10% → '+D.marginal_note.mw+' MW',
       showarrow:true, arrowcolor:'#e8632a', arrowhead:2, ax:34, ay:-26, font:{size:12, color:'#e8632a', family:'Space Grotesk'}}]}, CFG);
  // (6) TOP PRODUCERS
  const t10 = D.top.slice(0,10);
  Plotly.newPlot('pTop',[{x:t10.map(t=>t.output_kwh), y:t10.map((_,i)=>'#'+(i+1)), type:'bar', orientation:'h', showlegend:false,
    marker:{color:'#e8632a'}, hovertemplate:'#%{y}<br>%{x:,} '+T('ax_kwh')+'<extra></extra>'}],
    {...L, xaxis:{...L.xaxis, title:T('ax_kwh')}, yaxis:{...L.yaxis, autorange:'reversed'}, margin:{...L.margin, l:34}}, CFG);
  window._charts = true; observeResize();
}

// ✅ دالة إعادة تعيين الشارتات
function resetCharts() {
    const btn = $('#resetChartsBtn');
    if(!btn) return;
    btn.classList.add('resetting');
    
    const ids = ['pLor','pType','pSpa','pDist','pMarg','pTop'];
    ids.forEach(id => {
        const el = document.getElementById(id);
        if(el && el.data) {
            Plotly.relayout(el, {
                'xaxis.autorange': true,
                'yaxis.autorange': true,
                'xaxis.range': null,
                'yaxis.range': null
            });
        }
    });
    
    setTimeout(() => btn.classList.remove('resetting'), 600);
}

function restyleCharts(){ ['pLor','pType','pSpa','pDist','pMarg','pTop'].forEach(id=>{const e=document.getElementById(id); if(e&&e.data) Plotly.relayout(e, base());}); }
function resizeAll(){ ['pLor','pType','pSpa','pDist','pMarg','pTop'].forEach(id=>{const e=document.getElementById(id); if(e&&e.data) Plotly.Plots.resize(e);}); }
let _ro; function observeResize(){ if(_ro) return; _ro = new ResizeObserver(()=>{ if($('.view[data-view=analytics]').classList.contains('active')) resizeAll(); });
  $$('.cell .plot').forEach(p=>_ro.observe(p)); }

function renderGuide(){
  $('#readList').innerHTML = D.guide.map(g => {
    const o = g[LANG] || g.en; let txt = o.x;
    if(g.v !== '' && g.v !== undefined) txt = txt.replace('{v}', fmt(g.v));
    txt = txt.replace('{n}', fmt(D.marginal_note.n)).replace('{mw}', D.marginal_note.mw);
    return `<div class="it"><div class="ic">${g.icon}</div><div><div class="tt">${o.t}</div><div class="xx">${txt}</div></div></div>`;
  }).join('');
}
function renderLegend(){
  $('#legend').innerHTML = '<div style="font-size:10px;letter-spacing:1px;text-transform:uppercase;color:var(--muted);margin-bottom:6px">'+T('lg_title')+'</div>' +
    D.categories.map(c => `<div class="li"><span class="sw" style="background:${c.color}"></span>${catName(c.key)}<b>${fmt(c.count)}</b></div>`).join('');
}

let _geoLayer, _sliderMax;
const darkTiles  = L.tileLayer('https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',{subdomains:'abcd',maxZoom:19,attribution:'&copy; OSM &copy; CARTO'});
const lightTiles = L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{subdomains:'abcd',maxZoom:19,attribution:'&copy; OSM &copy; CARTO'});
function setBasemap(){ const light = document.documentElement.dataset.theme==='light';
  if(window._map){ window._map.eachLayer(l=>{ if(l._url && l._url.includes('basemaps.cartocdn')) window._map.removeLayer(l);}); (light?lightTiles:darkTiles).addTo(window._map); } }
function initMap(){
  if(window._map){ window._map.invalidateSize(); return; }
  const m = L.map('map',{zoomControl:false}).setView([48.148,11.566],14);
  window._map = m; setBasemap(); L.control.zoom({position:'bottomright'}).addTo(m);
  _sliderMax = Math.max(...window.BUILDINGS.features.map(f=>f.properties.o));
  _geoLayer = L.geoJSON(window.BUILDINGS, {
    style: f => { const c = D.cat_colors[f.properties.c]; return {color:c, weight:.6, fillColor:c, fillOpacity:.55}; },
    onEachFeature: (f, layer) => {
      layer.bindTooltip(`<b>${catName(f.properties.c)}</b><br>${fmt(f.properties.o)} kWh/yr`, {sticky:true, direction:'top'});
      layer.on('mouseover', e => e.target.setStyle({weight:2.2, fillOpacity:.85}));
      layer.on('mouseout',  e => _geoLayer.resetStyle(e.target));
    }
  }).addTo(m);
  try{ m.fitBounds(_geoLayer.getBounds(), {padding:[30,30]}); }catch(e){}
  const sl = $('#slider'); sl.max = _sliderMax; sl.value = 0; sl.step = Math.max(50, Math.round(_sliderMax/200));
  sl.oninput = () => applySlider(+sl.value);
  $('#baseBtn').onclick = () => { document.documentElement.dataset.theme = document.documentElement.dataset.theme==='light'?'dark':'light'; localStorage.setItem('solar-theme',document.documentElement.dataset.theme); setBasemap(); if(window._charts) restyleCharts(); };
  $('#fitBtn').onclick = () => m.fitBounds(_geoLayer.getBounds(), {padding:[30,30]});
  applySlider(0);
}
function applySlider(min){
  $('#slVal').textContent = fmt(min) + ' kWh'; let n=0, sum=0;
  _geoLayer.eachLayer(l => { const o = l.feature.properties.o, on = o >= min;
    l.setStyle({fillOpacity: on?.55:.04, opacity: on?1:.12, weight: on?.6:.2}); if(on){ n++; sum += o; } });
  $('#stN').textContent = fmt(n); $('#stOut').textContent = fmt(Math.round(sum/1000)) + ' MWh';
}
function refreshMapStats(){ if($('#slider')) applySlider(+$('#slider').value); }

$$('.lang button').forEach(b => b.onclick = () => setLang(b.dataset.l));
// ✅ ربط زر الـ Reset بالدالة
$('#resetChartsBtn').onclick = resetCharts;

renderKpis(); renderLegend(); renderGuide();
$('#heroNum').textContent = fmt(D.kpis[3].value);
$('#mEffV').textContent = D.glance.eff; $('#mTopV').textContent = D.glance.top20 + '%';
$('#mAvgV').textContent = D.glance.avg; $('#mExcV').textContent = D.glance.exc + '%';
setLang(LANG); showView('overview');
"""
with open("app.js", "w", encoding="utf-8") as f:
    f.write(JS)

# ═══════════════════════════════════════════════════════════
# 4) DASHBOARD.HTML (✅ تم تحديث الـ Sidebar وإضافة زر Reset)
# ═══════════════════════════════════════════════════════════
HTML = r"""<!DOCTYPE html>
<html lang="en" data-theme="dark">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Solar·München — PV Potential Observatory</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;700&family=DM+Sans:wght@400;500;600;700&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<link rel="stylesheet" href="styles.css">
</head>
<body>
<div class="app">
  <aside class="side">
    <div class="brand"><span class="b1">Solar</span><span class="b2">München</span></div>
    <div class="brand-sub">PV Potential Observatory</div>
    <nav class="nav">
      <button data-go="overview" class="active">
        <svg class="nav-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="3" y="3" width="7" height="7"/><rect x="14" y="3" width="7" height="7"/><rect x="3" y="14" width="7" height="7"/><rect x="14" y="14" width="7" height="7"/></svg>
        Overview
      </button>
      <button data-go="map">
        <svg class="nav-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M21 10c0 7-9 13-9 13s-9-6-9-13a9 9 0 0 1 18 0z"/><circle cx="12" cy="10" r="3"/></svg>
        Spatial Map
      </button>
      <button data-go="analytics">
        <svg class="nav-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><line x1="18" y1="20" x2="18" y2="10"/><line x1="12" y1="20" x2="12" y2="4"/><line x1="6" y1="20" x2="6" y2="14"/></svg>
        Analytics
      </button>
    </nav>
    <div class="side-foot">Data: OpenStreetMap<br>Radiation: EU PVGIS<br>Engine: Shapely + GeoPandas</div>
  </aside>
  <main class="main">
    <div class="topbar">
      <div class="crumb">Solar·München / <b>overview</b></div>
      <div class="actions">
        <div class="lang"><button data-l="en" class="on">EN</button><button data-l="de">DE</button></div>
        <button class="ibtn" id="themeBtn" title="Theme">◐</button>
      </div>
    </div>

    <section class="view active" data-view="overview">
      <div class="hero">
        <div>
          <div class="kicker" id="heroTag">ROOF-BY-ROOF ASSESSMENT</div>
          <h1 id="heroH">How much sun is<br>sleeping on these roofs?</h1>
          <p id="heroP">A spatial photovoltaic audit of every qualified building in Maxvorstadt.</p>
        </div>
        <div class="bignum"><div class="n serif" id="heroNum">0</div><div class="u" id="heroU">GWh / year unlockable</div></div>
      </div>
      <div class="kpis" id="kpiRow"></div>
      <div class="glance">
        <div class="card"><div class="kicker" id="glanceK" style="margin-bottom:12px">AT A GLANCE</div><div class="lead" id="leadTxt"></div></div>
        <div class="card"><div class="minis">
          <div class="mini"><div class="v" id="mEffV">—</div><div class="k" id="mEffK">median output density</div><div class="k" id="mEffU" style="color:var(--faint)">kWh per m²</div></div>
          <div class="mini"><div class="v" id="mTopV">—</div><div class="k" id="mTopK">top 20% of roofs</div><div class="k" id="mTopU" style="color:var(--faint)">of all energy</div></div>
          <div class="mini"><div class="v" id="mAvgV">—</div><div class="k" id="mAvgK">average per roof</div><div class="k" id="mAvgU" style="color:var(--faint)">kWh / year</div></div>
          <div class="mini"><div class="v" id="mExcV">—</div><div class="k" id="mExcK">excellent roofs</div><div class="k" id="mExcU" style="color:var(--faint)">> 10 MWh / year</div></div>
        </div></div>
      </div>
    </section>

    <section class="view" data-view="map">
      <div class="map-head"><div><div class="kicker" id="mapK">GEOSPATIAL</div><h2 id="mapH">Interactive Building Map</h2></div></div>
      <div class="map-stage">
        <div id="map"></div>
        <div class="float stats">
          <div class="row"><span id="sShowing">Showing</span><b id="stN">0</b></div>
          <div class="row"><span id="sOutLbl">their output</span><b id="stOut">0</b></div>
          <div class="row" style="color:var(--faint);font-size:11px"><span id="sOf">of</span>&nbsp;<span id="sSample">sampled roofs</span></div>
        </div>
        <div class="float legend" id="legend"></div>
        <div class="map-tools"><button id="baseBtn" title="Basemap">🗺️</button><button id="fitBtn" title="Fit">⤢</button></div>
        <div class="float slider">
          <label><span id="slLabel">Minimum annual output </span><b id="slVal">0</b></label>
          <input type="range" id="slider" min="0" value="0">
        </div>
      </div>
    </section>

    <section class="view" data-view="analytics">
      <div class="a-head">
        <div><div class="kicker" id="aK">DISTRIBUTION</div><h2 id="aH">Six readings that actually mean something</h2></div>
        <!-- ✅ زر Reset Charts -->
        <button class="reset-btn" id="resetChartsBtn" title="Reset all charts to default view">
          <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round">
            <polyline points="1 4 1 10 7 10"/>
            <path d="M3.51 15a9 9 0 1 0 2.13-9.36L1 10"/>
          </svg>
          <span>Reset Charts</span>
        </button>
      </div>
      <div class="a-grid">
        <div class="cell"><div class="ct" data-t="c_lorenz">Inequality — Lorenz & Gini</div><div class="cw" data-w="w_lorenz"></div><div id="pLor" class="plot"></div></div>
        <div class="cell"><div class="ct" data-t="c_type">Yield by building type</div><div class="cw" data-w="w_type"></div><div id="pType" class="plot"></div></div>
        <div class="cell"><div class="ct" data-t="c_spatial">Where the power lives</div><div class="cw" data-w="w_spatial"></div><div id="pSpa" class="plot"></div></div>
        <div class="cell"><div class="ct" data-t="c_dist">Output distribution</div><div class="cw" data-w="w_dist"></div><div id="pDist" class="plot"></div></div>
        <div class="cell"><div class="ct" data-t="c_marg">Marginal build-out</div><div class="cw" data-w="w_marg"></div><div id="pMarg" class="plot"></div></div>
        <div class="cell"><div class="ct" data-t="c_top">Top producers</div><div class="cw" data-w="w_top"></div><div id="pTop" class="plot"></div></div>
      </div>
      <div class="read-strip" id="readList"></div>
    </section>
  </main>
</div>
<script src="buildings.js"></script>
<script src="data.js"></script>
<script src="app.js"></script>
</body>
</html>"""
with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(HTML)

print("🎉 تم بنجاح! التعديلات الثلاثة طُبقت:")
print("   1. ✅ حل تحذير CRS (تحويل لـ EPSG:32632 ثم العودة)")
print("   2. ✅ استبدال النقاط بأيقونات SVG احترافية في الـ Sidebar")
print("   3. ✅ إضافة زر 'Reset Charts' لإعادة الشارتات لحالتها الأصلية")
print("\n📁 الملفات المُنشأة:")
for fn in ["dashboard.html","styles.css","app.js","data.js","buildings.js"]:
    if os.path.exists(fn):
        print(f"   ✅ {fn:18} {os.path.getsize(fn)/1024:7.1f} KB")